# 💊 Hap Sınıflandırma Projesi — V3 (Drive'dan Yükleme + Çapraz Doğrulama)
**Akış:** Drive Bağla → Veri Yükle → Modelleri Drive'dan Çek → Test Et → Cross-Validation → Karşılaştır

In [1]:
# ── HÜCRE 1: Drive Bağlantısı ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive bağlandı!')

Mounted at /content/drive
✅ Drive bağlandı!


In [2]:
# ── HÜCRE 2: Kütüphane Kurulumu ve İçe Aktarma ────────────────────────────
!pip install xgboost lightgbm -q

import os, cv2, numpy as np, random, warnings
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import cycle
from collections import Counter

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    mean_squared_error, mean_absolute_error,
    r2_score, roc_curve, auc, classification_report
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow.keras import layers, models, Model, Input
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import load_model
import joblib

warnings.filterwarnings('ignore')
print('✅ Tüm kütüphaneler başarıyla yüklendi!')

✅ Tüm kütüphaneler başarıyla yüklendi!


In [3]:
# ── HÜCRE 3: Veri Yolu ve Sınıflar ────────────────────────────────────────
veri_yolu = "/content/drive/MyDrive/YapayZeka_Hap_Projesi/archive/Drug Vision/Data Combined"
siniflar  = sorted([s for s in os.listdir(veri_yolu)
                    if os.path.isdir(os.path.join(veri_yolu, s))])

print(f"✅ Toplam {len(siniflar)} İlaç Sınıfı Bulundu:")
for i, s in enumerate(siniflar):
    print(f"  [{i}] {s}")

✅ Toplam 10 İlaç Sınıfı Bulundu:
  [0] Alaxan
  [1] Bactidol
  [2] Bioflu
  [3] Biogesic
  [4] DayZinc
  [5] Decolgen
  [6] Fish Oil
  [7] Kremil S
  [8] Medicol
  [9] Neozep


In [4]:
# ── HÜCRE 4: Veri Yükleme (Augmentation Yok — Ham Görüntüler) ─────────────
# Her çalıştırmada farklı bölünme için rastgele seed üretilir.
# Augmentation kaldırıldı: model gerçek veri dağılımıyla eğitilir.

SEED = random.randint(0, 9999)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f'🎲 Bu çalıştırmanın seed değeri: {SEED}')

print('📂 Veriler Drive\'dan yükleniyor... Lütfen bekleyin.')
X_cnn_orig, y_orig = [], []
BOYUT = (96, 96)

for sinif_adi in siniflar:
    yol = os.path.join(veri_yolu, sinif_adi)
    for resim_adi in os.listdir(yol):
        r = cv2.imread(os.path.join(yol, resim_adi))
        if r is None:
            continue
        r = cv2.resize(r, BOYUT)
        X_cnn_orig.append(cv2.cvtColor(r, cv2.COLOR_BGR2RGB).astype('float32') / 255.0)
        y_orig.append(sinif_adi)

X_cnn_orig = np.array(X_cnn_orig)
y_orig     = np.array(y_orig)
print(f'✅ Toplam orijinal görüntü: {len(y_orig)} adet.')

🎲 Bu çalıştırmanın seed değeri: 2445
📂 Veriler Drive'dan yükleniyor... Lütfen bekleyin.
✅ Toplam orijinal görüntü: 10000 adet.


In [5]:
# ── HÜCRE 5: Train / Test Bölme ────────────────────────────────────────────
# Augmentation yok — sadece ham görüntüler kullanılır.
# Her çalıştırmada farklı bölünme: SEED yukarıda rastgele belirlendi.

le = LabelEncoder()
le.fit(siniflar)
y_orig_num = le.transform(y_orig)

idx = np.arange(len(y_orig))
idx_train, idx_test = train_test_split(
    idx, test_size=0.2, random_state=SEED, stratify=y_orig
)

X_cnn_train = X_cnn_orig[idx_train]
X_cnn_test  = X_cnn_orig[idx_test]
y_train     = y_orig[idx_train]
y_test      = y_orig[idx_test]
y_train_num = y_orig_num[idx_train]
y_test_num  = y_orig_num[idx_test]
y_train_hot = tf.keras.utils.to_categorical(y_train_num, num_classes=len(siniflar))
y_test_hot  = tf.keras.utils.to_categorical(y_test_num,  num_classes=len(siniflar))

tum_sonuclar = {}
print(f'✅ İşlem Tamam. Eğitim Seti: {len(y_train)} | Test Seti: {len(y_test)}')

✅ İşlem Tamam. Eğitim Seti: 8000 | Test Seti: 2000


In [6]:
# ── HÜCRE 6: Metrik Hesaplama Fonksiyonu ──────────────────────────────────
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def metrik_hesapla(y_gercek_str, y_tahmin_str, model_adi, renk='Blues'):
    """
    Tüm metrikleri hesaplar, Confusion Matrix çizer.
    Giriş: string etiketler (sinif isimleri)
    """
    # String → sayısal
    y_g = le.transform(y_gercek_str)
    y_t = le.transform(np.array(y_tahmin_str))

    acc  = accuracy_score(y_g, y_t)
    mse  = mean_squared_error(y_g, y_t)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_g, y_t)
    r2   = r2_score(y_g, y_t)
    cm   = confusion_matrix(y_gercek_str, y_tahmin_str, labels=siniflar)
    cr   = classification_report(y_gercek_str, y_tahmin_str,
                                  target_names=siniflar, output_dict=True)

    print(f'\n{\'='*58}')
    print(f'  {model_adi}')
    print(f'{\'='*58}')
    print(f'  Accuracy   (Dogruluk)   : %{acc*100:.2f}')
    print(f'  R2 Skoru                : {r2:.4f}')
    print(f'  MSE                     : {mse:.4f}')
    print(f'  RMSE                    : {rmse:.4f}')
    print(f'  MAE                     : {mae:.4f}')
    print(f'  Kesinlik   (Precision)  : %{cr[\"weighted avg\"][\"precision\"]*100:.2f}')
    print(f'  Duyarlilik (Recall)     : %{cr[\"weighted avg\"][\"recall\"]*100:.2f}')
    print(f'  F1 Skoru                : %{cr[\"weighted avg\"][\"f1-score\"]*100:.2f}')
    print()
    print(classification_report(y_gercek_str, y_tahmin_str, target_names=siniflar))

    # Confusion Matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap=renk,
                xticklabels=siniflar, yticklabels=siniflar)
    plt.title(f'Confusion Matrix — {model_adi}', fontsize=13)
    plt.xlabel('Tahmin Edilen Sinif')
    plt.ylabel('Gercek Sinif')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    return {
        'acc'      : acc,
        'r2'       : r2,
        'mse'      : mse,
        'rmse'     : rmse,
        'mae'      : mae,
        'precision': cr['weighted avg']['precision'],
        'recall'   : cr['weighted avg']['recall'],
        'f1'       : cr['weighted avg']['f1-score'],
        'cm'       : cm
    }

print('Metrik fonksiyonu tanimlandi.')


SyntaxError: unexpected character after line continuation character (3355033457.py, line 22)

In [ ]:
# ── HÜCRE 7: Eğitilmiş Modelleri Drive'dan Yükle ──────────────────────────
model_klasoru = '/content/drive/MyDrive/Hap_Modelleri_V3'

print('📥 Modeller Drive\'dan yükleniyor...')
model_tl        = load_model(f'{model_klasoru}/mobilenet_v3_model.h5')
model_resnet    = load_model(f'{model_klasoru}/resnet50_hap_modeli.h5')
model_inception = load_model(f'{model_klasoru}/inceptionv3_hap_modeli.h5')
model_cnn_svm   = joblib.load(f'{model_klasoru}/cnn_svm_model.joblib')
model_cnn_knn   = joblib.load(f'{model_klasoru}/cnn_knn_model.joblib')
scaler_feat     = joblib.load(f'{model_klasoru}/scaler_feat.joblib')
le              = joblib.load(f'{model_klasoru}/label_encoder.joblib')

# Feature extractor (MobileNetV2'nin son softmax öncesi katmanı)
feat_model = Model(
    inputs=model_tl.input,
    outputs=model_tl.layers[-3].output
)

# Feature çıkarımı
print('🔍 Özellik vektörleri çıkarılıyor...')
X_feat_train = scaler_feat.transform(feat_model.predict(X_cnn_train, verbose=0))
X_feat_test  = scaler_feat.transform(feat_model.predict(X_cnn_test,  verbose=0))

print('✅ Tüm modeller ve özellik vektörleri hazır!')

In [ ]:
# ── HÜCRE 8: MobileNetV2 — Test Seti Degerlendirmesi ──────────────────────
print('\n' + '='*60)
print(' 1. MobileNetV2 — Test Seti Degerlendirmesi')
print('='*60)

y_score_tl    = model_tl.predict(X_cnn_test, verbose=0)
y_pred_tl_num = np.argmax(y_score_tl, axis=1)
y_pred_tl     = le.inverse_transform(y_pred_tl_num)

sonuc_tl = metrik_hesapla(y_test, y_pred_tl, 'MobileNetV2', renk='Blues')
tum_sonuclar['MobileNetV2'] = sonuc_tl


In [ ]:
# ── HÜCRE 9: MobileNetV2 — ROC Eğrisi ─────────────────────────────────────
y_test_bin    = label_binarize(y_test, classes=siniflar)
y_score_tl    = model_tl.predict(X_cnn_test, verbose=0)

fpr, tpr, roc_auc = {}, {}, {}
for i, sinif in enumerate(siniflar):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score_tl[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr['micro'], tpr['micro'], _ = roc_curve(y_test_bin.ravel(), y_score_tl.ravel())
roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])

plt.figure(figsize=(10, 7))
renkler = cycle(['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00',
                 '#a65628','#f781bf','#999999','#66c2a5','#fc8d62'])
for i, (sinif, renk) in enumerate(zip(siniflar, renkler)):
    plt.plot(fpr[i], tpr[i], color=renk, lw=1.5,
             label=f'{sinif} (AUC={roc_auc[i]:.3f})')
plt.plot(fpr['micro'], tpr['micro'], 'k--', lw=2,
         label=f'Mikro Ortalama (AUC={roc_auc["micro"]:.3f})')
plt.plot([0,1],[0,1],'gray',linestyle=':')
plt.xlabel('Yanlış Pozitif Oranı')
plt.ylabel('Doğru Pozitif Oranı')
plt.title('ROC Eğrisi — MobileNetV2 (Test Seti Öğrenci)', fontsize=13)
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── HÜCRE 10: CNN + kNN — Test Seti Degerlendirmesi ───────────────────────
print('\n' + '='*60)
print(' 2. CNN + kNN — Test Seti Degerlendirmesi')
print('='*60)

y_pred_knn    = model_cnn_knn.predict(X_feat_test)
y_pred_knn_str = le.inverse_transform(y_pred_knn)

sonuc_knn = metrik_hesapla(y_test, y_pred_knn_str, 'CNN + kNN', renk='Purples')
tum_sonuclar['CNN+kNN'] = sonuc_knn


In [ ]:
# ── HÜCRE 11: CNN + kNN — ROC Eğrisi ──────────────────────────────────────
y_score_knn = model_cnn_knn.predict_proba(X_feat_test)

fpr_k, tpr_k, roc_auc_k = {}, {}, {}
for i, sinif in enumerate(siniflar):
    fpr_k[i], tpr_k[i], _ = roc_curve(y_test_bin[:, i], y_score_knn[:, i])
    roc_auc_k[i] = auc(fpr_k[i], tpr_k[i])

fpr_k['micro'], tpr_k['micro'], _ = roc_curve(y_test_bin.ravel(), y_score_knn.ravel())
roc_auc_k['micro'] = auc(fpr_k['micro'], tpr_k['micro'])

plt.figure(figsize=(10, 7))
renkler = cycle(['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00',
                 '#a65628','#f781bf','#999999','#66c2a5','#fc8d62'])
for i, (sinif, renk) in enumerate(zip(siniflar, renkler)):
    plt.plot(fpr_k[i], tpr_k[i], color=renk, lw=1.5,
             label=f'{sinif} (AUC={roc_auc_k[i]:.3f})')
plt.plot(fpr_k['micro'], tpr_k['micro'], 'k--', lw=2,
         label=f'Mikro Ortalama (AUC={roc_auc_k["micro"]:.3f})')
plt.plot([0,1],[0,1],'gray',linestyle=':')
plt.xlabel('Yanlış Pozitif Oranı')
plt.ylabel('Doğru Pozitif Oranı')
plt.title('ROC Eğrisi — CNN + kNN (Test Seti Öğrenci)', fontsize=13)
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── HÜCRE 12: CNN + SVM — Test Seti Degerlendirmesi ───────────────────────
print('\n' + '='*60)
print(' 3. CNN + SVM — Test Seti Degerlendirmesi')
print('='*60)

y_pred_svm     = model_cnn_svm.predict(X_feat_test)
y_pred_svm_str = le.inverse_transform(y_pred_svm)

sonuc_svm = metrik_hesapla(y_test, y_pred_svm_str, 'CNN + SVM', renk='Greens')
tum_sonuclar['CNN+SVM'] = sonuc_svm


In [ ]:
# ── HÜCRE 13: CNN + SVM — ROC Eğrisi ──────────────────────────────────────
y_score_svm = model_cnn_svm.predict_proba(X_feat_test)

fpr_s, tpr_s, roc_auc_s = {}, {}, {}
for i, sinif in enumerate(siniflar):
    fpr_s[i], tpr_s[i], _ = roc_curve(y_test_bin[:, i], y_score_svm[:, i])
    roc_auc_s[i] = auc(fpr_s[i], tpr_s[i])

fpr_s['micro'], tpr_s['micro'], _ = roc_curve(y_test_bin.ravel(), y_score_svm.ravel())
roc_auc_s['micro'] = auc(fpr_s['micro'], tpr_s['micro'])

plt.figure(figsize=(10, 7))
renkler = cycle(['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00',
                 '#a65628','#f781bf','#999999','#66c2a5','#fc8d62'])
for i, (sinif, renk) in enumerate(zip(siniflar, renkler)):
    plt.plot(fpr_s[i], tpr_s[i], color=renk, lw=1.5,
             label=f'{sinif} (AUC={roc_auc_s[i]:.3f})')
plt.plot(fpr_s['micro'], tpr_s['micro'], 'k--', lw=2,
         label=f'Mikro Ortalama (AUC={roc_auc_s["micro"]:.3f})')
plt.plot([0,1],[0,1],'gray',linestyle=':')
plt.xlabel('Yanlış Pozitif Oranı')
plt.ylabel('Doğru Pozitif Oranı')
plt.title('ROC Eğrisi — CNN + SVM (Test Seti Öğrenci)', fontsize=13)
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── HÜCRE 14: ResNet50 — Test Seti Degerlendirmesi ────────────────────────
print('\n' + '='*60)
print(' 4. ResNet50 — Test Seti Degerlendirmesi')
print('='*60)

BOYUT_RN = (224, 224)
print('ResNet50 icin goruntuler hazirlaniyor...')
X_cnn_test_rn = np.array([
    cv2.resize((img * 255).astype('uint8'), BOYUT_RN).astype('float32')
    for img in X_cnn_test
])
X_cnn_test_rn = resnet_preprocess(X_cnn_test_rn)

y_score_rn    = model_resnet.predict(X_cnn_test_rn, verbose=0)
y_pred_rn_num = np.argmax(y_score_rn, axis=1)
y_pred_rn     = le.inverse_transform(y_pred_rn_num)

sonuc_rn = metrik_hesapla(y_test, y_pred_rn, 'ResNet50', renk='Oranges')
tum_sonuclar['ResNet50'] = sonuc_rn


In [ ]:
# ── HÜCRE 15: ResNet50 — ROC Eğrisi ───────────────────────────────────────
y_score_rn = model_resnet.predict(X_cnn_test_rn, verbose=0)

fpr_r, tpr_r, roc_auc_r = {}, {}, {}
for i, sinif in enumerate(siniflar):
    fpr_r[i], tpr_r[i], _ = roc_curve(y_test_bin[:, i], y_score_rn[:, i])
    roc_auc_r[i] = auc(fpr_r[i], tpr_r[i])

fpr_r['micro'], tpr_r['micro'], _ = roc_curve(y_test_bin.ravel(), y_score_rn.ravel())
roc_auc_r['micro'] = auc(fpr_r['micro'], tpr_r['micro'])

plt.figure(figsize=(10, 7))
renkler = cycle(['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00',
                 '#a65628','#f781bf','#999999','#66c2a5','#fc8d62'])
for i, (sinif, renk) in enumerate(zip(siniflar, renkler)):
    plt.plot(fpr_r[i], tpr_r[i], color=renk, lw=1.5,
             label=f'{sinif} (AUC={roc_auc_r[i]:.3f})')
plt.plot(fpr_r['micro'], tpr_r['micro'], 'k--', lw=2,
         label=f'Mikro Ortalama (AUC={roc_auc_r["micro"]:.3f})')
plt.plot([0,1],[0,1],'gray',linestyle=':')
plt.xlabel('Yanlış Pozitif Oranı')
plt.ylabel('Doğru Pozitif Oranı')
plt.title('ROC Eğrisi — ResNet50 (Test Seti Öğrenci)', fontsize=13)
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── HÜCRE 16: InceptionV3 — Test Seti Degerlendirmesi ────────────────────
print('\n' + '='*60)
print(' 5. InceptionV3 — Test Seti Degerlendirmesi')
print('='*60)

BOYUT_IV3 = (299, 299)
print('InceptionV3 icin goruntuler hazirlaniyor...')
X_cnn_test_iv3 = np.array([
    cv2.resize((img * 255).astype('uint8'), BOYUT_IV3).astype('float32')
    for img in X_cnn_test
])
X_cnn_test_iv3 = inception_preprocess(X_cnn_test_iv3)

y_score_iv3    = model_inception.predict(X_cnn_test_iv3, verbose=0)
y_pred_iv3_num = np.argmax(y_score_iv3, axis=1)
y_pred_iv3     = le.inverse_transform(y_pred_iv3_num)

sonuc_iv3 = metrik_hesapla(y_test, y_pred_iv3, 'InceptionV3', renk='Blues')
tum_sonuclar['InceptionV3'] = sonuc_iv3


In [ ]:
# ── HÜCRE 17: InceptionV3 — ROC Eğrisi ────────────────────────────────────
y_score_iv3 = model_inception.predict(X_cnn_test_iv3, verbose=0)

fpr_iv, tpr_iv, roc_auc_iv = {}, {}, {}
for i, sinif in enumerate(siniflar):
    fpr_iv[i], tpr_iv[i], _ = roc_curve(y_test_bin[:, i], y_score_iv3[:, i])
    roc_auc_iv[i] = auc(fpr_iv[i], tpr_iv[i])

fpr_iv['micro'], tpr_iv['micro'], _ = roc_curve(y_test_bin.ravel(), y_score_iv3.ravel())
roc_auc_iv['micro'] = auc(fpr_iv['micro'], tpr_iv['micro'])

plt.figure(figsize=(10, 7))
renkler = cycle(['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00',
                 '#a65628','#f781bf','#999999','#66c2a5','#fc8d62'])
for i, (sinif, renk) in enumerate(zip(siniflar, renkler)):
    plt.plot(fpr_iv[i], tpr_iv[i], color=renk, lw=1.5,
             label=f'{sinif} (AUC={roc_auc_iv[i]:.3f})')
plt.plot(fpr_iv['micro'], tpr_iv['micro'], 'k--', lw=2,
         label=f'Mikro Ortalama (AUC={roc_auc_iv["micro"]:.3f})')
plt.plot([0,1],[0,1],'gray',linestyle=':')
plt.xlabel('Yanlış Pozitif Oranı')
plt.ylabel('Doğru Pozitif Oranı')
plt.title('ROC Eğrisi — InceptionV3 (Test Seti Öğrenci)', fontsize=13)
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── HÜCRE 18: 5-Katlı Çapraz Doğrulama — TÜM MODELLER ────────────────────
# MobileNetV2, ResNet50, InceptionV3 → Manuel StratifiedKFold
# CNN + kNN, CNN + SVM                → cross_val_score (sklearn)

print('\n' + '='*60)
print(' 5-Fold Çapraz Doğrulama — TÜM MODELLER')
print('='*60)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_tum = {}  # tüm sonuçları burada toplayacağız

# ══════════════════════════════════════════════════════════════
# 1. MobileNetV2 — Manuel Fold (Keras modeli)
# ══════════════════════════════════════════════════════════════
print('\n📊 1. MobileNetV2 — 5-Fold Çapraz Doğrulama')
tl_scores = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cnn_orig, y_orig_num), 1):
    X_val_f = X_cnn_orig[val_idx]
    y_val_f = y_orig_num[val_idx]
    preds   = np.argmax(model_tl.predict(X_val_f, batch_size=64, verbose=0), axis=1)
    fold_acc = accuracy_score(y_val_f, preds)
    tl_scores.append(fold_acc)
    print(f'  Fold {fold}: %{fold_acc*100:.2f}')
tl_scores = np.array(tl_scores)
print(f'  Ortalama : %{tl_scores.mean()*100:.2f} ± %{tl_scores.std()*100:.2f}')
cv_tum['MobileNetV2'] = tl_scores

# ══════════════════════════════════════════════════════════════
# 2. ResNet50 — Manuel Fold (Keras modeli, 224x224 preprocessing)
# ══════════════════════════════════════════════════════════════
print('\n📊 2. ResNet50 — 5-Fold Çapraz Doğrulama')
rn_scores = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cnn_orig, y_orig_num), 1):
    X_val_f   = X_cnn_orig[val_idx]
    y_val_f   = y_orig_num[val_idx]
    # ResNet için 224x224 + preprocess
    X_val_rn  = np.array([
        cv2.resize((img*255).astype('uint8'), (224,224)).astype('float32')
        for img in X_val_f
    ])
    X_val_rn  = resnet_preprocess(X_val_rn)
    preds     = np.argmax(model_resnet.predict(X_val_rn, batch_size=32, verbose=0), axis=1)
    fold_acc  = accuracy_score(y_val_f, preds)
    rn_scores.append(fold_acc)
    print(f'  Fold {fold}: %{fold_acc*100:.2f}')
rn_scores = np.array(rn_scores)
print(f'  Ortalama : %{rn_scores.mean()*100:.2f} ± %{rn_scores.std()*100:.2f}')
cv_tum['ResNet50'] = rn_scores

# ══════════════════════════════════════════════════════════════
# 3. InceptionV3 — Manuel Fold (Keras modeli, 299x299 preprocessing)
# ══════════════════════════════════════════════════════════════
print('\n📊 3. InceptionV3 — 5-Fold Çapraz Doğrulama')
iv3_scores = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cnn_orig, y_orig_num), 1):
    X_val_f    = X_cnn_orig[val_idx]
    y_val_f    = y_orig_num[val_idx]
    # InceptionV3 için 299x299 + preprocess
    X_val_iv3  = np.array([
        cv2.resize((img*255).astype('uint8'), (299,299)).astype('float32')
        for img in X_val_f
    ])
    X_val_iv3  = inception_preprocess(X_val_iv3)
    preds      = np.argmax(model_inception.predict(X_val_iv3, batch_size=32, verbose=0), axis=1)
    fold_acc   = accuracy_score(y_val_f, preds)
    iv3_scores.append(fold_acc)
    print(f'  Fold {fold}: %{fold_acc*100:.2f}')
iv3_scores = np.array(iv3_scores)
print(f'  Ortalama : %{iv3_scores.mean()*100:.2f} ± %{iv3_scores.std()*100:.2f}')
cv_tum['InceptionV3'] = iv3_scores

# ══════════════════════════════════════════════════════════════
# 4. CNN + kNN — cross_val_score (sklearn modeli)
# ══════════════════════════════════════════════════════════════
# Tüm veri için feature çıkar (CV tüm veriyi kullanır)
print('\n📊 4. CNN + kNN — 5-Fold Çapraz Doğrulama')
X_feat_all   = feat_model.predict(X_cnn_orig, batch_size=64, verbose=0)
X_feat_all_s = scaler_feat.transform(X_feat_all)

knn_cv_model = KNeighborsClassifier(n_neighbors=5, metric='euclidean', n_jobs=-1)
knn_scores   = cross_val_score(knn_cv_model, X_feat_all_s, y_orig_num,
                               cv=skf, scoring='accuracy', n_jobs=-1)
for fold, s in enumerate(knn_scores, 1):
    print(f'  Fold {fold}: %{s*100:.2f}')
print(f'  Ortalama : %{knn_scores.mean()*100:.2f} ± %{knn_scores.std()*100:.2f}')
cv_tum['CNN + kNN'] = knn_scores

# ══════════════════════════════════════════════════════════════
# 5. CNN + SVM — cross_val_score (sklearn modeli)
# ══════════════════════════════════════════════════════════════
print('\n📊 5. CNN + SVM — 5-Fold Çapraz Doğrulama')
svm_cv_model = SVC(kernel='rbf', C=10, gamma='scale', probability=True)
svm_scores   = cross_val_score(svm_cv_model, X_feat_all_s, y_orig_num,
                               cv=skf, scoring='accuracy', n_jobs=-1)
for fold, s in enumerate(svm_scores, 1):
    print(f'  Fold {fold}: %{s*100:.2f}')
print(f'  Ortalama : %{svm_scores.mean()*100:.2f} ± %{svm_scores.std()*100:.2f}')
cv_tum['CNN + SVM'] = svm_scores

print('\n✅ Tüm modeller için çapraz doğrulama tamamlandı!')


In [ ]:
# ── HÜCRE 19: Çapraz Doğrulama Sonuçları — Bar Grafik (TÜM MODELLER) ──────
fold_labels = [f'Fold {i}' for i in range(1, 6)]
renkler_fold = ['#4878CF','#6ACC65','#D65F5F','#B47CC7','#C4AD66']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

model_cv_listesi = [
    ('MobileNetV2', tl_scores,  'Blues'),
    ('ResNet50',    rn_scores,  'Oranges'),
    ('InceptionV3', iv3_scores, 'Greens'),
    ('CNN + kNN',   knn_scores, 'Purples'),
    ('CNN + SVM',   svm_scores, 'YlOrRd'),
]

for ax, (isim, skorlar, cmap) in zip(axes, model_cv_listesi):
    bars = ax.bar(fold_labels, skorlar * 100,
                  color=renkler_fold,
                  edgecolor='black', linewidth=0.8, width=0.6)
    for bar, val in zip(bars, skorlar * 100):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 1.5,
                f'%{val:.1f}', ha='center', va='top',
                fontsize=9, fontweight='bold', color='white')
    ax.axhline(skorlar.mean() * 100, color='red', linestyle='--', linewidth=1.5,
               label=f'Ort: %{skorlar.mean()*100:.2f}')
    ax.set_title(f'{isim}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Doğruluk (%)')
    ax.set_ylim(0, 105)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

# 6. slot — özet karşılaştırma
ax = axes[5]
model_isimleri = [x[0] for x in model_cv_listesi]
ort_skorlar    = [x[1].mean() * 100 for x in model_cv_listesi]
std_skorlar    = [x[1].std()  * 100 for x in model_cv_listesi]
bar_renkleri   = ['#5B9BD5','#ED7D31','#70AD47','#B47CC7','#D65F5F']

bars = ax.barh(model_isimleri, ort_skorlar,
               xerr=std_skorlar, color=bar_renkleri,
               edgecolor='black', linewidth=0.8,
               capsize=4, height=0.5)
for bar, val in zip(bars, ort_skorlar):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'%{val:.2f}', va='center', fontsize=9, fontweight='bold')
ax.set_xlim(0, 112)
ax.set_xlabel('Ortalama Doğruluk (%)')
ax.set_title('CV Ortalama Karşılaştırması', fontsize=11, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.suptitle('5-Fold Çapraz Doğrulama — Tüm Modeller', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── HÜCRE 20: Çapraz Doğrulama Özet Tablosu — TÜM MODELLER ────────────────
print('\n' + '='*75)
print(' 5-Fold Çapraz Doğrulama Özet — Tüm Modeller')
print('='*75)

model_sirasi = ['MobileNetV2', 'ResNet50', 'InceptionV3', 'CNN + kNN', 'CNN + SVM']
skorlar_dict = {
    'MobileNetV2' : tl_scores,
    'ResNet50'    : rn_scores,
    'InceptionV3' : iv3_scores,
    'CNN + kNN'   : knn_scores,
    'CNN + SVM'   : svm_scores,
}

# Başlık
baslik = f"{'Fold':<10}" + ''.join([f'{m:>14}' for m in model_sirasi])
print(baslik)
print('-' * (10 + 14 * len(model_sirasi)))

# Fold satırları
for fold_i in range(5):
    satir = f'Fold {fold_i+1:<5}'
    for m in model_sirasi:
        satir += f'  %{skorlar_dict[m][fold_i]*100:>8.2f}  '
    print(satir)

print('-' * (10 + 14 * len(model_sirasi)))

# Ortalama satırı
satir_ort = f'Ortalama  '
for m in model_sirasi:
    satir_ort += f'  %{skorlar_dict[m].mean()*100:>8.2f}  '
print(satir_ort)

# Std sapma satırı
satir_std = f'Std Sapma '
for m in model_sirasi:
    satir_std += f'  %{skorlar_dict[m].std()*100:>8.2f}  '
print(satir_std)
print('='*75)

# En tutarlı model (düşük std)
en_tutarli = min(skorlar_dict, key=lambda m: skorlar_dict[m].std())
en_basarili = max(skorlar_dict, key=lambda m: skorlar_dict[m].mean())
print(f'\n🏆 En Yüksek Ortalama : {en_basarili} (%{skorlar_dict[en_basarili].mean()*100:.2f})')
print(f'🎯 En Tutarlı Model   : {en_tutarli}  (Std: %{skorlar_dict[en_tutarli].std()*100:.2f})')


In [ ]:
# ── HÜCRE 21: Model Karsilastirma Tablosu — Tum Metrikler ────────────────
modeller    = list(tum_sonuclar.keys())
acc_list    = [tum_sonuclar[m]['acc']  * 100 for m in modeller]
r2_list     = [tum_sonuclar[m]['r2']         for m in modeller]
mse_list    = [tum_sonuclar[m]['mse']        for m in modeller]
rmse_list   = [tum_sonuclar[m]['rmse']       for m in modeller]
mae_list    = [tum_sonuclar[m]['mae']        for m in modeller]
f1_list     = [tum_sonuclar[m]['f1']   * 100 for m in modeller]

print('\n' + '='*90)
print(f" {'Model':<15} {'Accuracy':>9} {'R2':>7} {'MSE':>7} {'RMSE':>7} {'MAE':>7} {'F1':>7}")
print('='*90)
for i, m in enumerate(modeller):
    print(f" {m:<15} %{acc_list[i]:>7.2f}  {r2_list[i]:>7.4f}  {mse_list[i]:>5.4f}  {rmse_list[i]:>5.4f}  {mae_list[i]:>5.4f}  %{f1_list[i]:>5.2f}")
print('='*90)

# ── Bar Grafik: Accuracy + F1 ──────────────────────────────────────────────
x      = np.arange(len(modeller))
width  = 0.3
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Sol: Accuracy + F1
ax = axes[0]
b1 = ax.bar(x - width/2, acc_list, width, label='Accuracy', color='#5B9BD5', edgecolor='black', lw=0.7)
b2 = ax.bar(x + width/2, f1_list,  width, label='F1 Skoru', color='#FFC000', edgecolor='black', lw=0.7)
for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
                f'{h:.1f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(modeller, fontsize=9, rotation=15)
ax.set_ylim(0, 115); ax.set_ylabel('Yuzde (%)'); ax.legend()
ax.set_title('Accuracy ve F1 Skoru Karsilastirmasi', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Sag: RMSE + MAE
ax = axes[1]
b3 = ax.bar(x - width/2, rmse_list, width, label='RMSE', color='#ED7D31', edgecolor='black', lw=0.7)
b4 = ax.bar(x + width/2, mae_list,  width, label='MAE',  color='#70AD47', edgecolor='black', lw=0.7)
for bars in [b3, b4]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(modeller, fontsize=9, rotation=15)
ax.set_ylabel('Hata Degeri'); ax.legend()
ax.set_title('RMSE ve MAE Hata Karsilastirmasi', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Tum Modeller — Kapsamli Metrik Karsilastirmasi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── R² Grafigi ─────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
renkler_r2 = ['#5B9BD5','#ED7D31','#70AD47','#FFC000','#B47CC7']
bars = plt.bar(modeller, r2_list, color=renkler_r2, edgecolor='black', lw=0.8)
for bar, val in zip(bars, r2_list):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.axhline(1.0, color='red', linestyle='--', lw=1.5, label='Ideal R²=1')
plt.ylim(0, 1.15)
plt.ylabel('R² Skoru')
plt.title('R² Skoru Karsilastirmasi (1\'e yakın = daha iyi)', fontweight='bold')
plt.legend(); plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── HÜCRE 20: En İyi Modeli Belirle ve Görselleştir ───────────────────────
en_iyi_adi  = max(tum_sonuclar, key=lambda m: tum_sonuclar[m]['f1'])
en_iyi_f1   = tum_sonuclar[en_iyi_adi]['f1'] * 100

print('\n' + '█'*60)
print(f'  🏆 EN İYİ MODEL: {en_iyi_adi}')
print(f'  F1 Skoru       : %{en_iyi_f1:.2f}')
print('█'*60)

# Tüm modellerin F1 skorlarını yatay bar grafiğiyle göster
f1_sirali   = sorted(tum_sonuclar.items(), key=lambda x: x[1]['f1'])
isimler     = [m[0] for m in f1_sirali]
degerler    = [m[1]['f1'] * 100 for m in f1_sirali]
renkler_bar = ['#FFC000' if m == en_iyi_adi else '#B0C4DE' for m in isimler]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(isimler, degerler, color=renkler_bar, edgecolor='black', linewidth=0.8, height=0.5)
for bar, val in zip(bars, degerler):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'%{val:.2f}', va='center', fontsize=10, fontweight='bold')
ax.set_xlim(0, 110)
ax.set_xlabel('F1 Skoru (%)', fontsize=11)
ax.set_title('Akademik Referans Modelleri — Başarı Karşılaştırması', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── HÜCRE 21: Derin Öğrenme Modelleri Karşılaştırma ───────────────────────
# MobileNetV2, ResNet50 ve InceptionV3'ü doğrudan karşılaştırır
dl_modeller  = ['MobileNetV2', 'ResNet50', 'InceptionV3']
dl_acc       = [tum_sonuclar[m]['acc'] * 100 for m in dl_modeller]
dl_f1        = [tum_sonuclar[m]['f1']  * 100 for m in dl_modeller]

print('\n📊 Derin Öğrenme Modelleri Karşılaştırması:')
print(f"  {'Model':<15} {'Doğruluk':>10} {'F1 Skoru':>10}")
print('  ' + '-'*37)
for m, a, f in zip(dl_modeller, dl_acc, dl_f1):
    print(f"  {m:<15} %{a:>8.2f}   %{f:>8.2f}")

kazanan_dl = dl_modeller[np.argmax(dl_f1)]
print(f'\n  🏅 Derin öğrenmede öne çıkan model: {kazanan_dl}')

In [ ]:
# ── HÜCRE 22: Kendi Fotoğrafını Test Et (Canlı Performans) ────────────────
from google.colab import files
print('\n📸 Sınav Vakti! Bilgisayarınızdan bir hap fotoğrafı yükleyin...')
uploaded = files.upload()

for fn in uploaded.keys():
    resim_bgr = cv2.imread(fn)
    resim_rgb = cv2.cvtColor(resim_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(4, 4))
    plt.imshow(resim_rgb)
    plt.title('Yüklenen Gerçek Fotoğraf')
    plt.axis('off')
    plt.show()

    # MobileNetV2 / Hibrit için 96x96
    r96      = cv2.resize(resim_bgr, (96, 96))
    r96_rgb  = cv2.cvtColor(r96, cv2.COLOR_BGR2RGB).astype('float32') / 255.0
    r96_batch = np.expand_dims(r96_rgb, axis=0)

    # ResNet50 için 224x224 + resnet_preprocess
    r224       = cv2.resize(resim_bgr, (224, 224))
    r224_rgb   = cv2.cvtColor(r224, cv2.COLOR_BGR2RGB).astype('float32')
    r224_pre   = resnet_preprocess(np.expand_dims(r224_rgb, axis=0))
    r224_batch = r224_pre

    # InceptionV3 için 299x299 + inception_preprocess
    r299       = cv2.resize(resim_bgr, (299, 299))
    r299_rgb   = cv2.cvtColor(r299, cv2.COLOR_BGR2RGB).astype('float32')
    r299_batch = inception_preprocess(np.expand_dims(r299_rgb, axis=0))

    ozellik_feat = scaler_feat.transform(feat_model.predict(r96_batch, verbose=0))

    tl_pred_num  = np.argmax(model_tl.predict(r96_batch, verbose=0))
    tl_pred      = siniflar[tl_pred_num]

    rn_pred_num  = np.argmax(model_resnet.predict(r224_batch, verbose=0))
    rn_pred      = siniflar[rn_pred_num]

    iv3_pred_num = np.argmax(model_inception.predict(r299_batch, verbose=0))
    iv3_pred     = siniflar[iv3_pred_num]

    cnn_knn_pred = str(model_cnn_knn.predict(ozellik_feat)[0])
    cnn_svm_pred = str(model_cnn_svm.predict(ozellik_feat)[0])

    print('\n🤖 LİTERATÜR MODELLERİNİN KARARLARI:')
    print(f"  {'─'*50}")
    print(f'  MobileNetV2 (Saf Derin Öğrenme) : {tl_pred.upper()}')
    print(f'  ResNet50    (Saf Derin Öğrenme) : {rn_pred.upper()}')
    print(f'  InceptionV3 (Saf Derin Öğrenme) : {iv3_pred.upper()}')
    print(f'  Hibrit Yaklaşım (CNN + kNN)     : {cnn_knn_pred.upper()}')
    print(f'  Klasik Hibrit  (CNN + SVM)      : {cnn_svm_pred.upper()}')
    print(f"  {'─'*50}")

    tum_tahminler = [tl_pred.lower(), rn_pred.lower(), iv3_pred.lower(),
                     cnn_knn_pred.lower(), cnn_svm_pred.lower()]
    kazanan = Counter(tum_tahminler).most_common(1)[0][0]
    print(f'\n🏆 YAPAY ZEKA KONSEYİ ORTAK KARARI: {kazanan.upper()}')

In [ ]:
# ── HÜCRE 23: Modelleri Drive'a Kaydet (İsteğe Bağlı) ─────────────────────
# Bu hücreyi SADECE modelleri yeniden eğittiyseniz çalıştırın.
# Drive'daki mevcut modellerin üzerine yazar!

kayit_klasoru = '/content/drive/MyDrive/Hap_Modelleri_V3'
if not os.path.exists(kayit_klasoru):
    os.makedirs(kayit_klasoru)

print(f"📦 Modeller '{kayit_klasoru}' klasörüne kaydediliyor...")

model_tl.save(f"{kayit_klasoru}/mobilenet_v3_model.h5")
print('  ✅ MobileNetV2 kaydedildi (.h5)')

joblib.dump(model_cnn_svm, f"{kayit_klasoru}/cnn_svm_model.joblib")
joblib.dump(model_cnn_knn, f"{kayit_klasoru}/cnn_knn_model.joblib")
print('  ✅ SVM ve kNN modelleri kaydedildi (.joblib)')

joblib.dump(scaler_feat, f"{kayit_klasoru}/scaler_feat.joblib")
joblib.dump(le, f"{kayit_klasoru}/label_encoder.joblib")
print('  ✅ Scaler ve LabelEncoder kaydedildi (.joblib)')

print('\n🎉 Tüm modeller Drive\'a güvenle yedeklendi!')